In [ ]:
# imports: data wrangling + plotting + stats, plus signal-processing bits for the theta/gamma stuff (tensorpac etc.)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.stats
from scipy.stats import shapiro, ttest_ind, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import ast
import os
from pandas.api.types import is_scalar

# --- SIGNAL PROCESSING & PAC ---
from scipy.signal import butter, sosfiltfilt, argrelextrema, hilbert
from tensorpac import Pac

In [ ]:
# load the LFP table -- one row per recording/session
df1 = pd.read_pickle(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/lfp_with_gamma_event_coupling.pkl")
data = []
for idx, row in df1.iterrows():
    animal_id = row['animal_id']
    s = row['session_id'].split("_")
    # session_id is like ID_date_time_A/B/C -> only keep the A sessions (open-field baseline)
    if s[3]=="A":
        data.append(row)

df = pd.DataFrame(data)

In [ ]:
import numpy as np
import pandas as pd

# helper: add up the power spectrum inside a given frequency band
def sum_band_power_from_cell(cell, band):
    """
    Extracts power and calculates sum within a band.
    Handles the case where data is a list containing a numpy array.
    """
    # 1. Handle potential nesting [array([...])]
    if isinstance(cell, list) and len(cell) > 0:
        data = np.array(cell[0]).flatten()
    elif isinstance(cell, np.ndarray):
        data = cell.flatten()
    else:
        return np.nan

    if len(data) == 0:
        return np.nan

    # 2. Reconstruct Frequencies
    # Based on your collection code: p_run = compute_mean_power_spectral_density(..., 1.5, fs=1250)
    # The frequency axis starts at 0 and goes up to fs/2 (625Hz).
    # Length 938 suggests it covers 0 to ~625Hz. 
    freqs = np.linspace(0, 625, len(data))

    # 3. Create Mask and Sum
    mask = (freqs >= band[0]) & (freqs <= band[1])
    mask &= ~((freqs >= 48) & (freqs <= 52))  # drop the 50 Hz notch region (filtering distorts 48-52 Hz)
    
    if not np.any(mask):
        return 0.0
    
    return np.nansum(data[mask])

# --- Main Logic ---

# Bands (Hz)
# the bands we care about (Hz)
THETA_BAND = (4.0, 12.0)
SLOW_GAMMA_BAND = (20.0, 39.0)
FAST_GAMMA_BAND = (40.0, 91.0)

lfp_cols = ["lfp_py_norm_run", "lfp_sr_norm_run", "lfp_py_norm_rest", "lfp_sr_norm_rest"]

# make band-power columns for each layer (py=pyramidal, sr=str. radiatum) x state (run/rest)
for col in lfp_cols:
    print(f"Processing {col}...")
    df[f"{col}_theta_sum"] = df[col].apply(lambda x: sum_band_power_from_cell(x, THETA_BAND))
    df[f"{col}_slow_gamma_sum"] = df[col].apply(lambda x: sum_band_power_from_cell(x, SLOW_GAMMA_BAND))
    df[f"{col}_fast_gamma_sum"] = df[col].apply(lambda x: sum_band_power_from_cell(x, FAST_GAMMA_BAND))

# Quick check
cols_to_view = [c for col in lfp_cols for c in (f"{col}_theta_sum", f"{col}_fast_gamma_sum")]
print(df[cols_to_view].head())

In [ ]:
# ==========================================
# 1. CONFIGURATION & STYLE
# ==========================================
import os
import ast
import numpy as np
import pandas as pd
import scipy.stats
from scipy.stats import shapiro, ttest_ind, mannwhitneyu
from scipy.signal import butter, sosfiltfilt, argrelextrema, hilbert
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
from pandas.api.types import is_scalar # Helper for safe scalar check
try:
    from tensorpac import Pac
except ImportError:
    pass # Assume user handles tensorpac in their environment

STAT_LEGEND_LOGS = []
# The per-frequency test is now a mixed model at the animal level (calculate_fdr_lmm).
# Nothing survives FDR correction, so the spectra are shown descriptively and the markers
# are suppressed; the statistics are still computed and logged for the legend.
SHOW_SPECTRAL_SIG_MARKERS = False
# 调整字体大小，适配 Neuron 期刊标准 (通常要求刻度 6-8pt，标签 8-10pt)
FONT_SIZE = 8
TITLE_SIZE = 9
TICK_SIZE = 7

TEXT_KWARGS = {'fontsize': FONT_SIZE, 'color': 'black'}
CACHE_FILE = "supp_analysis_cache.npz"

# 推荐的学术配色 (维持不变)
PALETTE = {'Control': 'blue', 'Experimental': 'red', 'Exp': 'red'}

plt.rcParams.update({
    'font.size': FONT_SIZE,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'sans-serif'],
    'axes.labelsize': FONT_SIZE,
    'axes.titlesize': TITLE_SIZE,
    'axes.titleweight': 'bold',  # 标题加粗，增加视觉层次
    'axes.titlepad': 8,          # 标题和图表增加呼吸感
    'xtick.labelsize': TICK_SIZE,
    'ytick.labelsize': TICK_SIZE,
    'legend.fontsize': TICK_SIZE,
    'axes.linewidth': 0.8,       # 坐标轴边框
    'xtick.major.width': 0.8,    
    'ytick.major.width': 0.8,
    'xtick.major.size': 3.5,     # 刻度线长度微调
    'ytick.major.size': 3.5,
    'xtick.direction': 'out',    
    'ytick.direction': 'out',
    'figure.dpi': 1200,
    'pdf.fonttype': 42,          # 保证导出PDF可编辑
    'ps.fonttype': 42,
    'axes.spines.top': False,    # 全局默认去掉上方和右侧边框
    'axes.spines.right': False
})

CONTROL_IDS = ['65165', '65091', '63383', '66539', '65622']
EXP_IDS = ['65588', '63385', '66538', '66537', '66922']
COMMON_FREQS = np.linspace(1, 151, 75)
FS = 1250

# ==========================================
# 2. DATA PROCESSING HELPERS 
# ==========================================

def collapse_cell_safe(x, delim=None, strategy="mean"):
    if is_scalar(x) or isinstance(x, pd.Timestamp): return x
    if isinstance(x, str):
        if delim and delim in x: vals = [t.strip() for t in x.split(delim) if t.strip() != ""]
        else:
            if x[:1] in "[({" and x[-1:] in "])}":
                try: x = ast.literal_eval(x)
                except: return x
            else: return x
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)): vals = list(x)
    elif "vals" not in locals(): return x
    if not vals: return np.nan
    nums = [float(v) for v in vals if str(v).replace('.','',1).isdigit()]
    if nums: return float(np.mean(nums))
    return vals[0]

def get_lfp_averages(df, animal_ids, column_name):
    animal_averages = {}
    for animal_id in animal_ids:
        animal_data = df[df['animal_id'] == animal_id][column_name]
        all_power_values = []
        for power_vector in animal_data:
            if isinstance(power_vector, list) and len(power_vector) > 0:
                data = np.array(power_vector[0]).flatten()
            elif isinstance(power_vector, np.ndarray):
                data = power_vector.flatten()
            elif isinstance(power_vector, pd.Series):
                data = power_vector.values
            else:
                continue
                
            if len(data) == 0 or np.all(np.isnan(data)):
                continue
                
            f_src = np.linspace(0, FS/2, len(data))
            interp_power = np.interp(COMMON_FREQS, f_src, data, left=np.nan, right=np.nan)
            all_power_values.append(interp_power)
            
        if all_power_values:
            try:
                stacked = np.vstack(all_power_values)
                animal_averages[animal_id] = np.nanmean(stacked, axis=0)
            except ValueError:
                animal_averages[animal_id] = np.full(len(COMMON_FREQS), np.nan)
        else:
            animal_averages[animal_id] = np.full(len(COMMON_FREQS), np.nan)
    return animal_averages

# grab the scalar metrics per session (average across channels)
def prepare_scalar_data(df, variables):
    data = []
    for idx, row in df.iterrows():
        aid = row['animal_id']
        cond = "Control" if aid in CONTROL_IDS else "Exp" if aid in EXP_IDS else None
        if cond:
            row_data = {"condition": cond, "animal_id": aid}
            for var in variables: row_data[var] = row[var]
            data.append(row_data)
    return pd.DataFrame(data).map(lambda v: collapse_cell_safe(v, delim=",", strategy="mean"))

# ==========================================
# 3. STATS & SIGNAL HELPERS
# ==========================================
# old per-session t-test/MWU -- kept around, but the figure now uses the mixed model instead
def run_statistical_test(ctrl_data, exp_data, label=""):
    c = ctrl_data[~np.isnan(ctrl_data)]
    e = exp_data[~np.isnan(exp_data)]
    n_c, n_e = len(c), len(e)
    
    if n_c < 3 or n_e < 3: 
        return 1.0, "N/A"
        
    # Normality check
    stat_c, p_c = shapiro(c)
    stat_e, p_e = shapiro(e)
    
    # Choose test based on normality
    if p_c > 0.05 and p_e > 0.05:
        stat, p_val = ttest_ind(c, e)
        test_name = "unpaired Student's t-test"
        df_val = n_c + n_e - 2
        stat_str = f"t({df_val}) = {stat:.3f}"
    else:
        stat, p_val = mannwhitneyu(c, e)
        test_name = "Mann-Whitney U test"
        stat_str = f"U = {stat:.1f}"
        
    # Format p-value string
    p_str = "p < 0.001" if p_val < 0.001 else f"p = {p_val:.4f}"
    
    # Generate Neuron-style legend text
    legend_text = f"{label}: {test_name}, {stat_str}, {p_str} (n_CR;DTA- = {n_c}, n_CR;DTA+ = {n_e})."
    
    return p_val, legend_text

# t-test at every frequency then FDR-correct, to mark which freqs differ
def calculate_fdr(control_powers, exp_powers, label="LFP Power Spectra"):
    p_values = []
    for idx in range(control_powers.shape[1]):
        if 48 <= COMMON_FREQS[idx] <= 52:   # skip the 50 Hz notch region
            p_values.append(np.nan)
            continue
        ctrl, ex = control_powers[:, idx], exp_powers[:, idx]
        ctrl, ex = ctrl[~np.isnan(ctrl)], ex[~np.isnan(ex)]
        if len(ctrl) >= 2 and len(ex) >= 2:
            p_values.append(scipy.stats.ttest_ind(ctrl, ex)[1])
        else:
            p_values.append(np.nan)
            
    p_vals = np.array(p_values)
    mask = ~np.isnan(p_vals)
    q = np.full_like(p_vals, np.nan)
    
    if np.any(mask): 
        q[mask] = multipletests(p_vals[mask], method='fdr_bh')[1]
        
    sig_freqs = COMMON_FREQS[q < 0.05]
    
    # Log the FDR results
    if len(sig_freqs) > 0:
        min_f, max_f = sig_freqs.min(), sig_freqs.max()
        STAT_LEGEND_LOGS.append(f"{label}: Significant differences found between {min_f:.1f}-{max_f:.1f} Hz (multiple unpaired t-tests with Benjamini-Hochberg FDR correction, q < 0.05).")
    else:
        STAT_LEGEND_LOGS.append(f"{label}: No significant differences found across frequencies (multiple unpaired t-tests with Benjamini-Hochberg FDR correction, q > 0.05).")
        
    return sig_freqs

# per-frequency test with animal as a random intercept, then FDR across frequencies.
# This replaces the old session-level t-test: sessions per animal are very unbalanced
# (1-10), so treating them as independent inflates significance -- the same reason the
# scalar panels use a mixed model. Keeps the statistical unit consistent across all of Fig. 3.
def calculate_fdr_lmm(lfp_df, label="LFP Power Spectra"):
    """lfp_df needs columns: animal_id, condition ('Control'/'Experimental'), average_lfp_power."""
    import statsmodels.formula.api as smf
    import warnings as _w

    mat = np.stack(lfp_df['average_lfp_power'].values)
    animals = lfp_df['animal_id'].values
    groups = np.where(lfp_df['condition'].isin(['Control']).values, 'Control', 'Exp')

    p_values = []
    for idx in range(mat.shape[1]):
        if 48 <= COMMON_FREQS[idx] <= 52:   # skip the 50 Hz notch region
            p_values.append(np.nan)
            continue
        d = pd.DataFrame({'y': mat[:, idx], 'animal_id': animals, 'g': groups}).dropna(subset=['y'])
        d = d[d['y'] > 0]
        d['y'] = np.log(d['y'])             # power is positive and right-skewed
        d['g'] = pd.Categorical(d['g'], categories=['Control', 'Exp'])
        if d['animal_id'].nunique() < 3 or len(d) < 6:
            p_values.append(np.nan)
            continue
        with _w.catch_warnings():
            _w.simplefilter('ignore')
            try:
                r = smf.mixedlm("y ~ g", d, groups=d['animal_id']).fit(reml=True)
                p_values.append(r.pvalues.get('g[T.Exp]', np.nan))
            except Exception:
                p_values.append(np.nan)

    p_vals = np.array(p_values, dtype=float)
    mask = ~np.isnan(p_vals)
    q = np.full_like(p_vals, np.nan)
    if np.any(mask):
        q[mask] = multipletests(p_vals[mask], method='fdr_bh')[1]

    sig_freqs = COMMON_FREQS[q < 0.05]
    n_sess, n_mice = len(lfp_df), lfp_df['animal_id'].nunique()
    if len(sig_freqs) > 0:
        # report contiguous significant runs, not just min-max
        runs, start = [], sig_freqs[0]
        for a, b in zip(sig_freqs[:-1], sig_freqs[1:]):
            if b - a > (COMMON_FREQS[1] - COMMON_FREQS[0]) * 1.5:
                runs.append((start, a)); start = b
        runs.append((start, sig_freqs[-1]))
        rtxt = ", ".join(f"{lo:.1f}-{hi:.1f} Hz" for lo, hi in runs)
        STAT_LEGEND_LOGS.append(
            f"{label}: significant differences at {rtxt} (linear mixed-effects model at each "
            f"frequency, animal as random intercept, Benjamini-Hochberg FDR across frequencies, "
            f"q < 0.05; n = {n_sess} recordings from {n_mice} mice; 48-52 Hz excluded).")
    else:
        STAT_LEGEND_LOGS.append(
            f"{label}: no significant differences across frequencies (linear mixed-effects model "
            f"at each frequency, animal as random intercept, Benjamini-Hochberg FDR, q > 0.05; "
            f"n = {n_sess} recordings from {n_mice} mice; 48-52 Hz excluded).")
    return sig_freqs


# def run_statistical_test(ctrl_data, exp_data):
#     c = ctrl_data[~np.isnan(ctrl_data)]
#     e = exp_data[~np.isnan(exp_data)]
#     if len(c) < 3 or len(e) < 3: return 1.0, "N/A"
#     stat_c, p_c = shapiro(c)
#     stat_e, p_e = shapiro(e)
#     if p_c > 0.05 and p_e > 0.05:
#         return ttest_ind(c, e)[1], 't-test'
#     return mannwhitneyu(c, e)[1], 'Mann-Whitney U'

# def calculate_fdr(control_powers, exp_powers):
#     p_values = []
#     for idx in range(control_powers.shape[1]):
#         ctrl, ex = control_powers[:, idx], exp_powers[:, idx]
#         ctrl, ex = ctrl[~np.isnan(ctrl)], ex[~np.isnan(ex)]
#         if len(ctrl) >= 2 and len(ex) >= 2:
#             p_values.append(scipy.stats.ttest_ind(ctrl, ex)[1])
#         else:
#             p_values.append(np.nan)
#     p_vals = np.array(p_values); mask = ~np.isnan(p_vals); q = np.full_like(p_vals, np.nan)
#     if np.any(mask): q[mask] = multipletests(p_vals[mask], method='fdr_bh')[1]
#     return COMMON_FREQS[q < 0.05]

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    sos = butter(order, [lowcut/nyq, highcut/nyq], btype='band', output='sos')
    return sosfiltfilt(sos, data)

def generate_normalized_spectrogram(signal, fs=1250, freq_step=2, num_phase_bins=12, start_phase=0, peak_at_deg=90):
    theta_filt = bandpass_filter(signal, 6, 12, fs)
    minima_idx = argrelextrema(theta_filt, np.less)[0]
    cycle_pairs = [(minima_idx[i], minima_idx[i+1]) for i in range(len(minima_idx)-1) if 80 <= (minima_idx[i+1]-minima_idx[i])/fs*1000 <= 170]
    if not cycle_pairs: raise ValueError("No theta cycles found.")
    phase = np.angle(hilbert(theta_filt))
    mean_trough_phase = np.angle(np.mean(np.exp(1j * phase[minima_idx])))
    aligned_phase = (phase - mean_trough_phase + np.deg2rad(peak_at_deg - 180)) % (2 * np.pi)
    freqs = np.arange(20, 121, freq_step)
    bin_edges = np.linspace(0, 2 * np.pi, num_phase_bins + 1)
    power_sum = np.zeros((len(freqs), num_phase_bins)); count = np.zeros((len(freqs), num_phase_bins))
    for f_idx, f in enumerate(freqs):
        sigma_f = f / 7.0; sigma_t = 1 / (2 * np.pi * sigma_f)
        t_wave = np.arange(-4 * sigma_t, 4 * sigma_t + 1/fs, 1/fs)
        wavelet = (1/np.sqrt(sigma_t*np.sqrt(np.pi))) * np.exp(-t_wave**2/(2*sigma_t**2)) * np.exp(2j*np.pi*f*t_wave)
        conv_pwr = np.abs(np.convolve(signal, wavelet, mode='same'))**2
        for start, end in cycle_pairs:
            c_phase = aligned_phase[start:end]; c_pwr = conv_pwr[start:end]
            b_idx = np.digitize(c_phase, bin_edges) - 1
            for b in range(num_phase_bins):
                mask = (b_idx == b)
                power_sum[f_idx, b] += np.sum(c_pwr[mask]); count[f_idx, b] += np.sum(mask)
    avg_p = np.divide(power_sum, count, where=count>0)
    norm_spec = avg_p / np.nanmean(avg_p, axis=1, keepdims=True)
    shift = int(start_phase / (360/num_phase_bins))
    shifted = np.roll(norm_spec, -shift, axis=1)
    return freqs, np.linspace(start_phase, start_phase+360, num_phase_bins+1), np.hstack((shifted, shifted[:, :1]))

# ==========================================
# 4. PLOTTING FUNCTIONS
# ==========================================
def setup_figure():
    # 1. 增加画布高度 (从 8.5 调整为 10.5)，给纵向留出充足的物理空间
    fig = plt.figure(figsize=(9, 10.5), layout='constrained')
    
    # 2. 增加 hspace=0.4 强制拉开 3 个主行之间的距离
    outer = gridspec.GridSpec(3, 1, height_ratios=[1, 1.2, 1.2], hspace=0.4)

    # --- Row 0 ---
    gs0 = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[0], width_ratios=[1.5, 2], wspace=0.2)
    ax_spectro = fig.add_subplot(gs0[0])
    sum_gs = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs0[1], wspace=0.6)
    sum_axes_list = [fig.add_subplot(sum_gs[i]) for i in range(3)]

    # --- Row 1 ---
    gs1 = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[1], width_ratios=[2.2, 1], wspace=0.25)
    mesh_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs1[0], wspace=0.3)
    ax_pcm_ctrl = fig.add_subplot(mesh_gs[0])
    ax_pcm_exp = fig.add_subplot(mesh_gs[1])
    coupling_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs1[1], wspace=0.55)
    ax_c_slow = fig.add_subplot(coupling_gs[0])
    ax_c_fast = fig.add_subplot(coupling_gs[1])

    # --- Row 2 ---
    gs2 = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[2], width_ratios=[2.2, 1], wspace=0.25)
    comod_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs2[0], wspace=0.3)
    ax_comod_ctrl = fig.add_subplot(comod_gs[0])
    ax_comod_exp = fig.add_subplot(comod_gs[1])
    rate_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs2[1], wspace=0.55)
    ax_r_slow = fig.add_subplot(rate_gs[0])
    ax_r_fast = fig.add_subplot(rate_gs[1])

    axes = {
        'spectro_1': ax_spectro,
        'lfp_lines': [ax_spectro], 
        'sum_axes': sum_axes_list,
        'pcm_ctrl': ax_pcm_ctrl,
        'pcm_exp': ax_pcm_exp,
        'coupling_slow': ax_c_slow,
        'coupling_fast': ax_c_fast,
        'comod_ctrl': ax_comod_ctrl,
        'comod_exp': ax_comod_exp,
        'rate_slow': ax_r_slow,
        'rate_fast': ax_r_fast
    }
    return fig, axes

# mean power spectrum with shading + little ticks where it's significant
def plot_lfp_lines(ax, df_lfp, sig_freqs):
    plot_data = []
    for _, row in df_lfp.iterrows():
        pwr = row['average_lfp_power']
        if isinstance(pwr, np.ndarray) and not np.all(np.isnan(pwr)):
            for f_idx, p in zip(COMMON_FREQS, pwr):
                if 48 <= f_idx <= 52:   # leave a gap over the 50 Hz notch (filtering distorts it)
                    continue
                plot_data.append({'condition': row['condition'], 'frequency': f_idx, 'power': p})
    
    if not plot_data: return 
    plot_df = pd.DataFrame(plot_data)
    
    sns.lineplot(data=plot_df, x='frequency', y='power', hue='condition', 
                 palette={'Control': PALETTE['Control'], 'Experimental': PALETTE['Exp']}, 
                 ax=ax, legend=False, linewidth=1.2, errorbar='se',
                 err_kws={'alpha': 0.15, 'linewidth': 0}) 
    
    ax.set_yscale('log') 
    ax.set_xlabel("Frequency (Hz)", **TEXT_KWARGS)
    ax.set_ylabel("Power (log)", **TEXT_KWARGS)
    ax.set_xlim(0, 90)
    ax.set_ylim(1e-3, 10**-.8)

    ax.grid(axis='y', which='major', linestyle='--', alpha=0.3, lw=0.6)

    # ---------------- 添加 Gamma 频段的背景遮罩和文本标注 ----------------
    # Slow Gamma (20-40 Hz)
    ax.axvspan(20, 40, color='gray', alpha=0.1, zorder=0, linewidth=0)
    # Fast Gamma (40-90 Hz)
    ax.axvspan(40, 90, color='gray', alpha=0.18, zorder=0, linewidth=0)
    # 50 Hz notch region excluded from comparisons
    ax.axvspan(48, 52, color='white', alpha=0.9, zorder=4, linewidth=0)
    
    # 动态计算文字的Y轴位置 (对数坐标下居中偏上)
    y_text_pos = 10**(np.log10(ax.get_ylim()[1]) - 0.4) 
    ax.text(30, y_text_pos, 'Slow $\gamma$', ha='center', va='center', fontsize=TICK_SIZE, alpha=0.8)
    ax.text(65, y_text_pos, 'Fast $\gamma$', ha='center', va='center', fontsize=TICK_SIZE, alpha=0.8)

    if SHOW_SPECTRAL_SIG_MARKERS and len(sig_freqs) > 0:
        y_pos = ax.get_ylim()[1] * 0.7 
        ax.scatter(sig_freqs, [y_pos] * len(sig_freqs), color='black', marker='s', s=6, zorder=5)
        
    sns.despine(ax=ax)

def _lfp_mixedlm_p(df, var_name):
    """LMM p for genotype with animal as random intercept; unit = session.
    Power/sum variables are log-transformed (positive, right-skewed)."""
    import statsmodels.formula.api as smf
    import warnings as _w
    d = df[['animal_id', 'condition', var_name]].copy()
    d['y'] = pd.to_numeric(d[var_name], errors='coerce')
    d = d.dropna(subset=['y'])
    if ('sum' in var_name) or ('power' in var_name.lower()):
        d = d[d['y'] > 0]; d['y'] = np.log(d['y'])
    d['g'] = pd.Categorical(d['condition'], categories=['Control', 'Exp'])
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        try:
            return smf.mixedlm("y ~ g", d, groups=d['animal_id']).fit(reml=True).pvalues.get('g[T.Exp]', np.nan)
        except Exception:
            return np.nan


def plot_scalar_comparison(ax, df, var_name, title):
    """SuperPlot: faint box (session distribution) + sessions colored by animal +
    per-animal mean (large dots) + mean +/- SEM across mice.
    Statistical unit = animal; significance = linear mixed model
    (session ~ condition + (1|animal)). Sessions per animal vary a lot, so the
    animal (not the recording) is the unit."""
    order = ['Control', 'Exp']
    xmap = {'Control': 0, 'Exp': 1}
    d = df.dropna(subset=[var_name]).copy()

    # faint box = session-level spread
    sns.boxplot(data=d, x='condition', y=var_name, ax=ax, order=order, hue='condition',
                legend=False, palette={"Control": PALETTE['Control'], "Exp": PALETTE['Exp']},
                width=0.45, linewidth=0.8, showfliers=False, boxprops={'alpha': 0.25, 'zorder': 1},
                whiskerprops={'alpha': 0.5}, capprops={'alpha': 0.5})

    # sessions colored by animal + per-animal mean (large dot)
    for g in order:
        sub = d[d['condition'] == g]
        animals = sorted(sub['animal_id'].unique())
        for ai, a in enumerate(animals):
            yy = pd.to_numeric(sub[sub['animal_id'] == a][var_name], errors='coerce').dropna().values
            if len(yy) == 0:
                continue
            xx = np.random.normal(xmap[g], 0.07, len(yy))
            ax.scatter(xx, yy, s=4, color=PALETTE[g], alpha=0.40, linewidth=0, zorder=2)
            ax.scatter(xmap[g], np.mean(yy), s=30, color=PALETTE[g], edgecolor='white', linewidth=0.6, zorder=4)

    # mean +/- SEM across mice
    for g in order:
        am = d[d['condition'] == g].groupby('animal_id')[var_name].mean().values
        if len(am) > 0:
            ax.errorbar(xmap[g], np.mean(am), yerr=np.std(am) / np.sqrt(len(am)),
                        fmt='_', color='k', capsize=3, markersize=11, zorder=5, elinewidth=1.0)

    ax.set_ylabel(title, **TEXT_KWARGS)
    ax.set_xlabel('')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation=45, ha='right', fontsize=TICK_SIZE)
    sns.despine(ax=ax)
    ax.set_ylim(bottom=0)  # LFP metrics are non-negative -> start y-axis at 0

    # significance from mixed-effects model + legend log with n
    p_val = _lfp_mixedlm_p(d, var_name)
    n_sess = len(d)
    n_mice = d['animal_id'].nunique()
    STAT_LEGEND_LOGS.append(
        f"{title}: LMM (animal random effect), p = {p_val:.4g} "
        f"(n = {n_sess} sessions from {n_mice} mice).")
    if pd.notna(p_val):
        ax.set_title(f"p = {p_val:.3f}", fontsize=TICK_SIZE)   # always show the p-value
    if pd.notna(p_val) and p_val < 0.05:
        y_min, y_max = d[var_name].min(), d[var_name].max()
        y_range = y_max - y_min
        bar_h = y_range * 0.05
        top_y = y_max + bar_h * 2
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*'
        ax.plot([0, 0, 1, 1], [y_max + bar_h, top_y, top_y, y_max + bar_h], color='black', lw=0.8, zorder=10)
        ax.text(0.5, top_y + bar_h * 0.2, sig, ha='center', va='bottom', fontsize=FONT_SIZE, color='black')
        ax.set_ylim(bottom=0, top=top_y + bar_h * 3)


# just the slow/fast gamma guide lines + labels on the heatmaps
def add_gamma_bands_to_heatmap(ax, text_x_pos, color='white'):
    """为热力图添加水平频率辅助线和文字"""
    ax.axhline(20, color=color, linestyle='--', linewidth=0.6, alpha=0.7)
    ax.axhline(40, color=color, linestyle='--', linewidth=0.6, alpha=0.7)
    ax.axhline(90, color=color, linestyle='--', linewidth=0.6, alpha=0.7)
    
    ax.text(text_x_pos, 30, 'Slow $\gamma$', color=color, fontsize=TICK_SIZE, va='center', ha='left', alpha=0.95)
    ax.text(text_x_pos, 65, 'Fast $\gamma$', color=color, fontsize=TICK_SIZE, va='center', ha='left', alpha=0.95)

# ==========================================
# 5. MAIN EXECUTION
# ==========================================

# (注意：此处保留你代码的运行逻辑。请确保外围环境中的 df 和 df1 已被正常加载。)

# the heatmaps take forever, so cache them and just reload next time
if os.path.exists(CACHE_FILE):
    print("--- Loading Heatmaps from Cache ---")
    cache = np.load(CACHE_FILE)
    freqs, phases, norm_spectrogram = cache['f1'], cache['p1'], cache['s1']
    freqs_exp, phases_exp, norm_spectrogram_exp = cache['f2'], cache['p2'], cache['s2']
    xpac, xpac_exp = cache['x1'], cache['x2']
else:
    print("--- Computing Heatmaps (One-time) ---")
    # 此处的 df1 需要在运行环境中存在
    s_c = df1['lfp_py'][6][0].values if isinstance(df1['lfp_py'][6][0], pd.Series) else np.array(df1['lfp_py'][6][0]).flatten()
    s_e = df1['lfp_py'][85][1].values if isinstance(df1['lfp_py'][85][1], pd.Series) else np.array(df1['lfp_py'][85][1]).flatten()
    freqs, phases, norm_spectrogram = generate_normalized_spectrogram(s_c, FS)
    freqs_exp, phases_exp, norm_spectrogram_exp = generate_normalized_spectrogram(s_e, FS)
    p_c = Pac(idpac=(5,0,0), f_pha=(1,20,1,0.2), f_amp=(12,150,10,2))
    xpac  = p_c.filterfit(FS, s_c).mean(axis=-1)
    p_e = Pac(idpac=(5,0,0), f_pha=(1,20,1,0.2), f_amp=(12,150,10,1))
    xpac_exp = p_e.filterfit(FS, s_e).mean(axis=-1)
    # 此处存储时修正了未定义的 xpac_mean_ctrl，改为 xpac
    np.savez(CACHE_FILE, f1=freqs, p1=phases, s1=norm_spectrogram, f2=freqs_exp, p2=phases_exp, s2=norm_spectrogram_exp, x1=xpac, x2=xpac_exp)

# 计算 PAC
phase_freqs = (1, 20, 1, 0.2)
amp_freqs_ctrl = (12, 150, 10, 2)
amp_freqs_exp = (12, 150, 10, 1) 

p_control = Pac(idpac=(5, 0, 0), f_pha=phase_freqs, f_amp=amp_freqs_ctrl)
p_exp = Pac(idpac=(5, 0, 0), f_pha=phase_freqs, f_amp=amp_freqs_exp)

fig, axes = setup_figure()

# --- A. LFP Lines (Session-based) ---
lfp_cols = ["lfp_py_norm_run"]

for i, col in enumerate(lfp_cols):
    session_data = []
    
    for _, row in df.iterrows():
        aid = row['animal_id']
        cond = 'Control' if aid in CONTROL_IDS else 'Experimental' if aid in EXP_IDS else 'Unknown'
        
        if cond != 'Unknown':
            power_vector = row[col]
            
            if isinstance(power_vector, (list, np.ndarray, pd.Series)):
                if isinstance(power_vector, list) and len(power_vector) > 0:
                    data = np.array(power_vector[0]).flatten()
                else:
                    data = np.array(power_vector).flatten()
                
                if len(data) > 0 and not np.all(np.isnan(data)):
                    f_src = np.linspace(0, FS/2, len(data))
                    interp_power = np.interp(COMMON_FREQS, f_src, data, left=np.nan, right=np.nan)
                    session_data.append({'condition': cond, 'animal_id': aid,
                                         'average_lfp_power': interp_power})

    lfp_df = pd.DataFrame(session_data)
    
    if not lfp_df.empty:
        # per-frequency LMM (animal random intercept) + BH FDR -- same statistical
        # unit as every other panel in Fig. 3
        plot_lfp_lines(axes['lfp_lines'][i], lfp_df,
                       calculate_fdr_lmm(lfp_df, label="Main LFP Power Spectra (pyramidal layer)"))

# --- B. Sum Power Comparison ---
# Slow gamma is measured in stratum radiatum, where the CA3-driven rhythm is
# generated (Reviewer #2, point 1); theta and fast gamma stay on the pyramidal
# layer. Panel labels carry the layer so the figure is unambiguous.
sum_vars = ["lfp_py_norm_run_theta_sum", "lfp_sr_norm_run_slow_gamma_sum", "lfp_py_norm_run_fast_gamma_sum"]
sum_titles = ["Theta (pyr.)", "Slow $\gamma$ (SR)", "Fast $\gamma$ (pyr.)"]
sum_df = prepare_scalar_data(df, sum_vars)
for i, var in enumerate(sum_vars):
    plot_scalar_comparison(axes['sum_axes'][i], sum_df, var, sum_titles[i])

# --- C. Spectrograms (Row 1) ---
pcm = axes['pcm_ctrl'].pcolormesh(phases, freqs, norm_spectrogram, shading='gouraud', cmap='viridis', vmin=0.5, vmax=2, rasterized=True)
cbar1 = plt.colorbar(pcm, ax=axes['pcm_ctrl'])
cbar1.ax.tick_params(labelsize=TICK_SIZE)
cbar1.outline.set_linewidth(0.6) 

axes['pcm_ctrl'].set_xticks([145, 145+180])
axes['pcm_ctrl'].set_xticklabels([0, 180], fontsize=TICK_SIZE)
axes['pcm_ctrl'].set_xlabel('Theta Phase (deg)', **TEXT_KWARGS)
axes['pcm_ctrl'].set_ylabel('Frequency (Hz)', **TEXT_KWARGS)
axes['pcm_ctrl'].set_title('CR;DTA-', **TEXT_KWARGS)

pcm_exp = axes['pcm_exp'].pcolormesh(phases_exp, freqs_exp, norm_spectrogram_exp, shading='gouraud', cmap='viridis', vmin=0.5, vmax=2, rasterized=True)
cbar2 = plt.colorbar(pcm_exp, ax=axes['pcm_exp'])
cbar2.ax.tick_params(labelsize=TICK_SIZE)
cbar2.outline.set_linewidth(0.6)

axes['pcm_exp'].set_xticks([145, 145+180])
axes['pcm_exp'].set_xticklabels([0, 180], fontsize=TICK_SIZE)
axes['pcm_exp'].set_xlabel('Theta Phase (deg)', **TEXT_KWARGS)
axes['pcm_exp'].set_ylabel('Frequency (Hz)', **TEXT_KWARGS)
axes['pcm_exp'].set_title('CR;DTA+', **TEXT_KWARGS)

add_gamma_bands_to_heatmap(axes['pcm_ctrl'], text_x_pos=axes['pcm_ctrl'].get_xlim()[0] + 5)
add_gamma_bands_to_heatmap(axes['pcm_exp'], text_x_pos=axes['pcm_exp'].get_xlim()[0] + 5)

# --- D. Comodulograms (Row 2) ---
pac_ctrl_2d = np.squeeze(xpac)
if pac_ctrl_2d.ndim > 2:
    pac_ctrl_2d = pac_ctrl_2d.mean(axis=-1)

pac_exp_2d = np.squeeze(xpac_exp)
if pac_exp_2d.ndim > 2:
    pac_exp_2d = pac_exp_2d.mean(axis=-1)

ax_ctrl = axes['comod_ctrl']
plt.sca(ax_ctrl)
p_control.comodulogram(pac_ctrl_2d, cmap='viridis', plotas='imshow', title='', vmin=0, vmax=0.35, colorbar=False)
mappable_ctrl = ax_ctrl.images[0]
cbar_c = plt.colorbar(mappable_ctrl, ax=ax_ctrl)
cbar_c.ax.tick_params(labelsize=TICK_SIZE)
cbar_c.outline.set_linewidth(0.6)
ax_ctrl.set_xlabel('Phase Frequency (Hz)', **TEXT_KWARGS)
ax_ctrl.set_ylabel('Amplitude Frequency (Hz)', **TEXT_KWARGS)
ax_ctrl.tick_params(axis='both', which='major', labelsize=TICK_SIZE)

ax_exp = axes['comod_exp']
plt.sca(ax_exp)
p_exp.comodulogram(pac_exp_2d, cmap='viridis', plotas='imshow', title='', vmin=0, vmax=0.35, colorbar=False)
mappable_exp = ax_exp.images[0]
cbar_e = plt.colorbar(mappable_exp, ax=ax_exp)
cbar_e.ax.tick_params(labelsize=TICK_SIZE)
cbar_e.outline.set_linewidth(0.6)
ax_exp.set_xlabel('Phase Frequency (Hz)', **TEXT_KWARGS)
ax_exp.set_ylabel('Amplitude Frequency (Hz)', **TEXT_KWARGS)
ax_exp.tick_params(axis='both', which='major', labelsize=TICK_SIZE)

add_gamma_bands_to_heatmap(ax_ctrl, text_x_pos=2)
add_gamma_bands_to_heatmap(ax_exp, text_x_pos=2)

# --- E. Coupling & Event Rate ---
scalar_data = prepare_scalar_data(df, ['slow_theta_gamma_coupling_sr', 'fast_theta_gamma_coupling_py', 
                                       'slow_event_rate_sr', 'fast_event_rate_py'])

plot_scalar_comparison(axes['coupling_slow'], scalar_data, 'slow_theta_gamma_coupling_sr', 'Slow-$\gamma$ vector length (SR)')
plot_scalar_comparison(axes['coupling_fast'], scalar_data, 'fast_theta_gamma_coupling_py', 'Fast-$\gamma$ vector length (pyr.)')
plot_scalar_comparison(axes['rate_slow'], scalar_data, 'slow_event_rate_sr', 'Slow-$\gamma$ rate (events/s, SR)')
plot_scalar_comparison(axes['rate_fast'], scalar_data, 'fast_event_rate_py', 'Fast-$\gamma$ rate (events/s, pyr.)')

# ==========================================
# OUTPUT
# ==========================================
# 动态匹配路径，防止本地找不到路径报错
save_path = r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Figures_neuron_report_raw'
if not os.path.exists(save_path):
    os.makedirs(save_path, exist_ok=True)

plt.savefig(os.path.join(save_path, 'lfp.pdf'), 
            transparent=True, dpi=1200,
            bbox_inches='tight')

plt.show()
# ==========================================
# GENERATE FIGURE LEGEND TEXT
# ==========================================
print("\n" + "="*50)
print("📚 AUTO-GENERATED STATISTICAL LEGEND FOR NEURON")
print("="*50)
print("Data are represented as mean ± SEM. Statistical significance was assessed using normal-based or non-parametric tests based on the Shapiro-Wilk test for normality.\n")

for i, log in enumerate(STAT_LEGEND_LOGS):
    # Map the list order roughly to the figure panels (A, B, C, etc.)
    panel_letter = chr(65 + i) 
    print(f"({panel_letter}) {log}")
print("="*50 + "\n")

In [ ]:
# ==========================================================================
# STRATUM RADIATUM CONTROL ANALYSIS  (Reviewer #2, point 1)
# --------------------------------------------------------------------------
# Valero's point: slow gamma is layer-specific and is maximal in stratum
# radiatum (SR), where CA3 input terminates -- so measuring it on the
# pyramidal-layer (PY) channel may under-represent it, and the criterion for
# channel selection was never stated.
#
# We have the SR channel for the same sessions, so we repeat every scalar
# measure of Fig. 3 on SR and compare it side-by-side with PY, using the same
# LMM (metric ~ genotype + (1|animal); power measures log-transformed).
#
# HOW THE SR CHANNEL WAS DEFINED (see Lfp.ipynb, find_lower_four_channel_per_group):
#   PY = the pyramidal-layer centre channel = the middle site, by depth, of the
#        shank's sites carrying ripple-detected pyramidal units.
#   SR = the 4th site below PY in the depth-ordered site list of the same shank.
# Probe: Cambridge NeuroTech ASSY-236-F -- 10-11 sites per shank in two columns
# 16.5 um apart, 15 um vertical step, so the sites of one shank span only
# 135-150 um in depth. The 4th site below PY therefore sits 60 um below the
# pyramidal-layer centre, which is the deepest site the probe geometry allows.
# This is superficial/proximal stratum radiatum rather than mid-SR -- a genuine
# limitation of the probe, and it is stated as such in the Methods.
#
# Note the SR coverage is smaller than PY: not every shank had an identified
# SR channel, so n drops from 59 sessions / 10 mice (PY) to 46 / 9 (SR;
# animal 66922 contributes no SR session).
# ==========================================================================

SR_STAT_LOGS = []

# --- SR versions of every scalar in Fig. 3 -------------------------------
SR_PAIRS = [
    ("Theta power",           "lfp_py_norm_run_theta_sum",      "lfp_sr_norm_run_theta_sum"),
    ("Slow-gamma power",      "lfp_py_norm_run_slow_gamma_sum", "lfp_sr_norm_run_slow_gamma_sum"),
    ("Fast-gamma power",      "lfp_py_norm_run_fast_gamma_sum", "lfp_sr_norm_run_fast_gamma_sum"),
    ("Slow-gamma event rate", "slow_event_rate_py",             "slow_event_rate_sr"),
    ("Fast-gamma event rate", "fast_event_rate_py",             "fast_event_rate_sr"),
    ("Slow-gamma coupling",   "slow_theta_gamma_coupling_py",   "slow_theta_gamma_coupling_sr"),
    ("Fast-gamma coupling",   "fast_theta_gamma_coupling_py",   "fast_theta_gamma_coupling_sr"),
]

sr_all_vars = [v for _, a, b in SR_PAIRS for v in (a, b)]
layer_df = prepare_scalar_data(df, sr_all_vars)


def _lmm_full(d, var_name):
    """Same model as _lfp_mixedlm_p, but also returns beta, n sessions, n mice."""
    import statsmodels.formula.api as smf
    import warnings as _w
    dd = d[['animal_id', 'condition', var_name]].copy()
    dd['y'] = pd.to_numeric(dd[var_name], errors='coerce')
    dd = dd.dropna(subset=['y'])
    if ('sum' in var_name) or ('power' in var_name.lower()):
        dd = dd[dd['y'] > 0]
        dd['y'] = np.log(dd['y'])
    dd['g'] = pd.Categorical(dd['condition'], categories=['Control', 'Exp'])
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        try:
            r = smf.mixedlm("y ~ g", dd, groups=dd['animal_id']).fit(reml=True)
            return (r.pvalues.get('g[T.Exp]', np.nan), r.params.get('g[T.Exp]', np.nan),
                    len(dd), dd['animal_id'].nunique())
        except Exception:
            return (np.nan, np.nan, len(dd), dd['animal_id'].nunique())


print("=" * 92)
print("PYRAMIDAL LAYER vs STRATUM RADIATUM  --  LMM: metric ~ genotype + (1|animal)")
print("=" * 92)
print(f"{'Metric':<22} | {'PY p':>8} {'beta':>7} {'n':>8} | {'SR p':>8} {'beta':>7} {'n':>8} | dir")
print("-" * 92)
for name, v_py, v_sr in SR_PAIRS:
    p1, b1, n1, m1 = _lmm_full(layer_df, v_py)
    p2, b2, n2, m2 = _lmm_full(layer_df, v_sr)
    same = "same" if (pd.notna(b1) and pd.notna(b2) and np.sign(b1) == np.sign(b2)) else "OPPOSITE"
    s = lambda p: '*' if pd.notna(p) and p < 0.05 else ' '
    print(f"{name:<22} | {p1:>8.4f}{s(p1)}{b1:>7.3f} {n1:>3}/{m1:<4} | "
          f"{p2:>8.4f}{s(p2)}{b2:>7.3f} {n2:>3}/{m2:<4} | {same}")
    SR_STAT_LOGS.append(
        f"{name}: pyramidal layer p = {p1:.4g} (beta = {b1:+.3f}, n = {n1} recordings / {m1} mice); "
        f"stratum radiatum p = {p2:.4g} (beta = {b2:+.3f}, n = {n2} recordings / {m2} mice).")
print("=" * 92)

# raw slow-gamma power in each layer -- shows slow gamma IS larger in SR,
# which is the direct answer to "slow gamma is not clearly present"
_sg = layer_df.groupby('condition')[['lfp_py_norm_run_slow_gamma_sum',
                                     'lfp_sr_norm_run_slow_gamma_sum']].mean()
print("\nMean normalised slow-gamma (20-39 Hz) power, 48-52 Hz excluded:")
print(f"  pyramidal layer : CR;DTA- = {_sg.loc['Control'].iloc[0]:.4f}   CR;DTA+ = {_sg.loc['Exp'].iloc[0]:.4f}")
print(f"  stratum radiatum: CR;DTA- = {_sg.loc['Control'].iloc[1]:.4f}   CR;DTA+ = {_sg.loc['Exp'].iloc[1]:.4f}")
print(f"  -> slow gamma is {_sg.loc['Control'].iloc[1] / _sg.loc['Control'].iloc[0]:.2f}x larger in SR "
      f"than in PY in controls, as expected for a CA3-driven rhythm.\n")


# ==========================================================================
# SUPPLEMENTARY FIGURE: the whole of Fig. 3's quantification, on SR
# ==========================================================================
fig_sr = plt.figure(figsize=(11, 7), layout='constrained')
outer_sr = gridspec.GridSpec(2, 1, figure=fig_sr, hspace=0.35)

gs_top = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer_sr[0],
                                          width_ratios=[1.5, 2], wspace=0.2)
ax_sr_spec = fig_sr.add_subplot(gs_top[0])
gs_sum = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_top[1], wspace=0.6)
ax_sr_sum = [fig_sr.add_subplot(gs_sum[i]) for i in range(3)]

gs_bot = gridspec.GridSpecFromSubplotSpec(1, 4, subplot_spec=outer_sr[1], wspace=0.75)
ax_sr_scalar = [fig_sr.add_subplot(gs_bot[i]) for i in range(4)]

# --- SR power spectrum, same treatment as Fig. 3A ---
sr_rows = []
for _, row in df.iterrows():
    aid = row['animal_id']
    cond = 'Control' if aid in CONTROL_IDS else 'Experimental' if aid in EXP_IDS else None
    if cond is None:
        continue
    pv = row['lfp_sr_norm_run']
    if not isinstance(pv, (list, np.ndarray, pd.Series)):
        continue
    data = np.array(pv[0]).flatten() if (isinstance(pv, list) and len(pv) > 0) else np.array(pv).flatten()
    if len(data) == 0 or np.all(np.isnan(data)):
        continue
    f_src = np.linspace(0, FS / 2, len(data))
    sr_rows.append({'condition': cond, 'animal_id': aid,
                    'average_lfp_power': np.interp(COMMON_FREQS, f_src, data, left=np.nan, right=np.nan)})

sr_lfp_df = pd.DataFrame(sr_rows)
if not sr_lfp_df.empty:
    _n0 = len(STAT_LEGEND_LOGS)
    plot_lfp_lines(ax_sr_spec, sr_lfp_df,
                   calculate_fdr_lmm(sr_lfp_df, label="LFP Power Spectra (stratum radiatum)"))
    SR_STAT_LOGS.extend(STAT_LEGEND_LOGS[_n0:])
    ax_sr_spec.set_title('Stratum radiatum\n(60 $\\mu$m below pyramidal layer centre)',
                         fontsize=TICK_SIZE, color='black')

# --- SR band power + SR coupling/event rate ---
for ax, var, title in zip(ax_sr_sum,
                          ["lfp_sr_norm_run_theta_sum", "lfp_sr_norm_run_slow_gamma_sum",
                           "lfp_sr_norm_run_fast_gamma_sum"],
                          ["Theta (SR)", r"Slow $\gamma$ (SR)", r"Fast $\gamma$ (SR)"]):
    plot_scalar_comparison(ax, layer_df, var, title)

for ax, var, title in zip(ax_sr_scalar,
                          ['slow_theta_gamma_coupling_sr', 'fast_theta_gamma_coupling_sr',
                           'slow_event_rate_sr', 'fast_event_rate_sr'],
                          [r'Slow-$\gamma$ vector length (SR)', r'Fast-$\gamma$ vector length (SR)',
                           r'Slow-$\gamma$ rate (events/s, SR)', r'Fast-$\gamma$ rate (events/s, SR)']):
    plot_scalar_comparison(ax, layer_df, var, title)

plt.savefig(os.path.join(save_path, 'lfp_stratum_radiatum.pdf'),
            transparent=True, dpi=1200, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print("STRATUM RADIATUM SUPPLEMENTARY FIGURE -- STATS FOR LEGEND")
print("=" * 60)
for log in SR_STAT_LOGS:
    print("  " + log)
print("=" * 60)
